# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya Exploration with `mlcroissant`
This notebook provides a structured approach for loading and exploring a dataset using the `mlcroissant` library. We will load, overview, and process the FAIR² Mental Health Survey Dataset as defined by its Croissant schema.

### Dataset Source
The dataset is provided via a Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`. This step fetches the Croissant schema and initializes the dataset object.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata.to_json()

print(f"Dataset Name: {metadata['name']}")
print(f"Description: {metadata['description']}")
print(f"Published: {metadata['datePublished']}")

## 2. Data Overview
Review available record sets, fields, and their IDs. The Croissant schema organizes the data as record sets. We enumerate these, referencing their `@id` for further processing.

In [ ]:
# Get record set IDs from metadata
record_sets_raw = metadata.get('recordSet', []) if 'recordSet' in metadata else []
record_set_ids = []
for rs in record_sets_raw:
    if isinstance(rs, dict) and '@id' in rs:
        record_set_ids.append(rs['@id'])

# If there are no record sets (as in metadata above), list distributions as record sources
if len(record_set_ids) == 0:
    # In Croissant, some datasets expose distributions or fileObjects as record sets
    distributions = metadata.get('distribution', [])
    print("Record set IDs not explicitly listed in metadata['recordSet']. Using distribution @id as record set IDs.")
    record_set_ids = [d['@id'] for d in distributions if isinstance(d, dict) and '@id' in d]

print("Available record set @ids:")
for rsid in record_set_ids:
    print(f"  - {rsid}")

# Preview content from each record set using mlcroissant
for record_set_id in record_set_ids:
    print(f"\nSample records from RecordSet: {record_set_id}")
    try:
        for i, x in enumerate(dataset.records(record_set=record_set_id)):
            if i >= 3: break
            print(x)
    except Exception as e:
        print(f"[Error loading records from {record_set_id}]: {e}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id` values from the overview above.
All references are made using `@id`.

We'll load each available record set into a pandas DataFrame.

In [ ]:
# Extract data from each record set (by @id)
dataframes = {}

for record_set_id in record_set_ids:
    print(f"Loading records from {record_set_id}")
    try:
        records = list(dataset.records(record_set=record_set_id))
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Columns for {record_set_id}: {df.columns.tolist()}")
        print(df.head())
    except Exception as e:
        print(f"[Error loading records from {record_set_id}]: {e}")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps: filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

We select a numeric field (e.g., GAD-7 score, PHQ-9 score, or PSQ score) and demonstrate filtering, normalization, and grouping.

In [ ]:
# Example using the first record set
record_set_id = record_set_ids[0] if len(record_set_ids) > 0 else None
df = dataframes.get(record_set_id, pd.DataFrame())

# Find numeric field candidates
numeric_fields = []
for col in df.columns:
    if df[col].dtype in ['int64', 'float64'] or pd.api.types.is_numeric_dtype(df[col]):
        numeric_fields.append(col)

print("Numeric fields detected:", numeric_fields)

# Choose a field such as 'gad7_score', 'phq9_score', or 'psq_score' if available
preferred_fields = ['gad7_score', 'phq9_score', 'psq_score']
numeric_field_id = None
for pf in preferred_fields:
    if pf in numeric_fields:
        numeric_field_id = pf
        break
if numeric_field_id is None and len(numeric_fields) > 0:
    numeric_field_id = numeric_fields[0]

if numeric_field_id is not None and record_set_id is not None:
    threshold = df[numeric_field_id].mean() if not pd.isnull(df[numeric_field_id].mean()) else 10
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold}:")
    print(filtered_df.head())

    filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Attempt grouping by 'village' or any demographic column
    group_field_id = 'village' if 'village' in df.columns else None
    if group_field_id:
        grouped_df = filtered_df.groupby(group_field_id).mean()
        print(f"Grouped data by {group_field_id}:")
        print(grouped_df.head())
else:
    print("No suitable numeric fields found for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset. Let's plot the normalized scores distribution if available.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Visualization of normalized numeric field
if numeric_field_id is not None and f"{numeric_field_id}_normalized" in filtered_df.columns:
    plt.figure(figsize=(8, 5))
    sns.histplot(filtered_df[f"{numeric_field_id}_normalized"], kde=True, bins=30)
    plt.title(f"Distribution of Normalized {numeric_field_id} Scores (filtered)")
    plt.xlabel(f"{numeric_field_id}_normalized")
    plt.ylabel("Count")
    plt.show()

    # If village grouping was performed
    if group_field_id and group_field_id in filtered_df.columns:
        plt.figure(figsize=(10, 6))
        sns.boxplot(x=filtered_df[group_field_id], y=filtered_df[numeric_field_id])
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion
We successfully loaded and explored the FAIR² Mental Health Survey Dataset from Kilifi County, Kenya using `mlcroissant`. Key steps included referencing all Croissant entities by their `@id`, extracting record sets, exploring numeric demographic and mental health scores, and visualizing their distributions.

This structure serves as a reproducible standard for AI-ready datasets, demonstrating both the FAIR principles and operational readiness for further machine learning or statistical analysis.